[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/01_gpu_fundamentals/01.6_serving_implications/lab.ipynb) [![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.cloud/github/harshuljain13/llm-inference-at-scale/blob/master/content/01_gpu_fundamentals/01.6_serving_implications/lab.ipynb)# Lab 1.6: Serving ImplicationsThis lab connects GPU hardware constraints to real serving decisions.You will predict latency/throughput from first principles, evaluatetensor parallelism tradeoffs, compare quantization vs. multi-GPU scaling,and build an end-to-end config recommender.

In [ ]:
import numpy as npimport matplotlib.pyplot as pltfrom dataclasses import dataclassfrom typing import Optional@dataclassclass GPU:    name: str    hbm_gb: float    bandwidth_gb_s: float    flops_tflops: float    cost_per_hour: float@dataclassclass Model:    name: str    params_b: float    layers: int    hidden_dim: int    num_heads: int    num_kv_heads: int# Reference GPUsA100_80 = GPU("A100-80GB", 80, 2039, 312, 3.50)H100_80 = GPU("H100-80GB", 80, 3350, 989, 5.50)A10G = GPU("A10G-24GB", 24, 600, 125, 1.00)RTX_6000 = GPU("RTX Pro 6000", 96, 1500, 300, 2.50)# Reference Models (approximate)LLAMA_7B = Model("Llama-2-7B", 7, 32, 4096, 32, 32)LLAMA_13B = Model("Llama-2-13B", 13, 40, 5120, 40, 40)LLAMA_70B = Model("Llama-2-70B", 70, 80, 8192, 64, 8)MIXTRAL_8x7B = Model("Mixtral-8x7B", 46.7, 32, 4096, 32, 8)print("GPU and Model definitions loaded.")

## Exercise 1: Predict Latency and ThroughputGiven a model and GPU, compute:- **Memory footprint** at FP16- **Decode token latency** (memory-bandwidth bound)- **Maximum throughput** (tokens/sec) at batch=1 and optimal batch

In [ ]:
def predict_serving_metrics(model: Model, gpu: GPU, batch_size: int = 1, precision_bytes: int = 2):    """Predict latency and throughput from first principles."""    # Weight memory    weight_bytes = model.params_b * 1e9 * precision_bytes    weight_gb = weight_bytes / 1e9    # KV cache per token per layer (key + value)    kv_per_token_per_layer = 2 * model.hidden_dim * precision_bytes    # For GQA: scale by kv_heads/num_heads    gqa_ratio = model.num_kv_heads / model.num_heads    kv_per_token = kv_per_token_per_layer * model.layers * gqa_ratio    # Decode latency: read all weights once per token (memory-bound)    # Time = bytes_to_read / bandwidth    bytes_per_step = weight_bytes + batch_size * kv_per_token * 1024  # assume 1024 ctx    decode_latency_ms = (bytes_per_step / (gpu.bandwidth_gb_s * 1e9)) * 1000    # Throughput    tokens_per_sec = batch_size / (decode_latency_ms / 1000)    # Compute-bound crossover batch size    # When compute time > memory time, we're compute-bound    # FLOPs per token ~= 2 * params (forward pass)    flops_per_token = 2 * model.params_b * 1e9    compute_time_per_token = flops_per_token / (gpu.flops_tflops * 1e12)    memory_time = weight_bytes / (gpu.bandwidth_gb_s * 1e9)    crossover_batch = int(np.ceil(memory_time / compute_time_per_token))    return {        "weight_gb": weight_gb,        "fits_on_gpu": weight_gb < gpu.hbm_gb * 0.85,  # 85% usable        "decode_latency_ms": decode_latency_ms,        "tokens_per_sec": tokens_per_sec,        "crossover_batch": crossover_batch,        "max_throughput_tps": crossover_batch / (decode_latency_ms / 1000 * crossover_batch / batch_size),    }# Run predictionsfor model in [LLAMA_7B, LLAMA_13B, LLAMA_70B]:    for gpu in [A10G, A100_80, H100_80]:        r = predict_serving_metrics(model, gpu)        fit = "✓" if r["fits_on_gpu"] else "✗"        print(f"{model.name:15s} on {gpu.name:12s} | {fit} | "              f"Weight: {r['weight_gb']:.1f}GB | "              f"Latency: {r['decode_latency_ms']:.1f}ms | "              f"TPS@b1: {r['tokens_per_sec']:.0f} | "              f"Crossover batch: {r['crossover_batch']}")    print()

## Exercise 2: Tensor Parallelism Bandwidth GainsCompare TP=1 vs TP=2 vs TP=4 for a 70B model.TP splits weight reads across GPUs but adds all-reduce communication overhead.

In [ ]:
def tp_analysis(model: Model, gpu: GPU, tp_degrees: list[int]):    """Analyze latency vs TP degree with NVLink overhead."""    results = []    weight_bytes = model.params_b * 1e9 * 2  # FP16    for tp in tp_degrees:        # Each GPU reads 1/TP of the weights        bytes_per_gpu = weight_bytes / tp        read_time_ms = (bytes_per_gpu / (gpu.bandwidth_gb_s * 1e9)) * 1000        # All-reduce overhead: 2*(tp-1)/tp * message_size / nvlink_bw        # Message per layer: hidden_dim * 2 bytes * batch        nvlink_bw = 600e9  # 600 GB/s NVLink (A100/H100)        msg_bytes = model.hidden_dim * 2 * model.layers        allreduce_time_ms = (2 * (tp - 1) / tp * msg_bytes / nvlink_bw) * 1000 if tp > 1 else 0        total_ms = read_time_ms + allreduce_time_ms        speedup = None        results.append({"tp": tp, "read_ms": read_time_ms, "allreduce_ms": allreduce_time_ms,                        "total_ms": total_ms, "cost_mult": tp})    # Compute speedups relative to TP=1    base = results[0]["total_ms"]    for r in results:        r["speedup"] = base / r["total_ms"]        r["efficiency"] = r["speedup"] / r["tp"]  # ideal=1.0    return resultstp_results = tp_analysis(LLAMA_70B, H100_80, [1, 2, 4, 8])print(f"{'TP':>3} | {'Read(ms)':>9} | {'AllReduce(ms)':>13} | {'Total(ms)':>9} | {'Speedup':>7} | {'Efficiency':>10}")print("-" * 70)for r in tp_results:    print(f"{r['tp']:>3} | {r['read_ms']:>9.2f} | {r['allreduce_ms']:>13.4f} | "          f"{r['total_ms']:>9.2f} | {r['speedup']:>7.2f}x | {r['efficiency']:>9.1%}")# Plotfig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))tps = [r["tp"] for r in tp_results]ax1.bar(range(len(tps)), [r["total_ms"] for r in tp_results], color="#dbeafe", edgecolor="#000")ax1.set_xticks(range(len(tps)))ax1.set_xticklabels([f"TP={t}" for t in tps])ax1.set_ylabel("Decode latency (ms)")ax1.set_title("70B FP16 on H100: Latency vs TP")ax2.plot(tps, [r["efficiency"] for r in tp_results], "o-", color="#2563eb")ax2.axhline(1.0, ls="--", color="#64748b", alpha=0.5)ax2.set_xlabel("TP degree")ax2.set_ylabel("Parallel efficiency")ax2.set_title("Scaling efficiency (1.0 = ideal)")ax2.set_ylim(0, 1.1)plt.tight_layout()plt.savefig("tp_analysis.png", dpi=100, bbox_inches="tight")plt.show()print("Saved tp_analysis.png")

## Exercise 3: Quantization vs More GPUs (Cost Analysis)Question: Is it cheaper to quantize a model (fewer bytes, some quality loss)or add more GPUs (full precision, higher cost)?

In [ ]:
def cost_comparison(model: Model, gpu: GPU, hours: float = 1.0):    """Compare quantization vs multi-GPU for serving cost."""    configs = []    precisions = [        ("FP16", 2, 1.0),    # bytes_per_param, quality_factor        ("INT8", 1, 0.98),        ("INT4", 0.5, 0.95),    ]    for name, bpp, quality in precisions:        weight_gb = model.params_b * 1e9 * bpp / 1e9        gpus_needed = int(np.ceil(weight_gb / (gpu.hbm_gb * 0.80)))        cost = gpus_needed * gpu.cost_per_hour * hours        # Throughput scales with fewer bytes to read        bytes_to_read = model.params_b * 1e9 * bpp        latency_ms = (bytes_to_read / (gpu.bandwidth_gb_s * 1e9 * gpus_needed)) * 1000        tps = 1000 / latency_ms        configs.append({            "precision": name, "weight_gb": weight_gb, "gpus": gpus_needed,            "cost_hr": cost, "latency_ms": latency_ms, "tps": tps,            "quality": quality, "cost_per_1k_tokens": cost / (tps * 3600) * 1000        })    return configsprint("=== Llama-70B on H100-80GB ===")print(f"{'Precision':>9} | {'Weight':>7} | {'GPUs':>4} | {'$/hr':>6} | {'Latency':>8} | {'TPS':>6} | {'$/1k tok':>8} | {'Quality':>7}")print("-" * 80)for c in cost_comparison(LLAMA_70B, H100_80):    print(f"{c['precision']:>9} | {c['weight_gb']:>5.1f}GB | {c['gpus']:>4} | "          f"${c['cost_hr']:>5.2f} | {c['latency_ms']:>6.1f}ms | {c['tps']:>5.0f} | "          f"${c['cost_per_1k_tokens']:>7.5f} | {c['quality']:>6.0%}")print("\n=== Llama-70B on A10G-24GB (budget option) ===")print(f"{'Precision':>9} | {'Weight':>7} | {'GPUs':>4} | {'$/hr':>6} | {'Latency':>8} | {'TPS':>6} | {'$/1k tok':>8}")print("-" * 75)for c in cost_comparison(LLAMA_70B, A10G):    print(f"{c['precision']:>9} | {c['weight_gb']:>5.1f}GB | {c['gpus']:>4} | "          f"${c['cost_hr']:>5.2f} | {c['latency_ms']:>6.1f}ms | {c['tps']:>5.0f} | "          f"${c['cost_per_1k_tokens']:>7.5f}")# Visualizationfig, ax = plt.subplots(figsize=(8, 5))configs_h100 = cost_comparison(LLAMA_70B, H100_80)configs_a10g = cost_comparison(LLAMA_70B, A10G)x = np.arange(3)w = 0.35ax.bar(x - w/2, [c["cost_per_1k_tokens"] * 1000 for c in configs_h100], w, label="H100-80GB", color="#dbeafe", edgecolor="#000")ax.bar(x + w/2, [c["cost_per_1k_tokens"] * 1000 for c in configs_a10g], w, label="A10G-24GB", color="#dcfce7", edgecolor="#000")ax.set_xticks(x)ax.set_xticklabels(["FP16", "INT8", "INT4"])ax.set_ylabel("Cost per 1M tokens ($)")ax.set_title("70B Model: Quantization vs GPU Tier Cost")ax.legend()plt.tight_layout()plt.savefig("quant_vs_gpu_cost.png", dpi=100, bbox_inches="tight")plt.show()

## Exercise 4: End-to-End Config RecommenderGiven a model name and target latency, output the recommendedserving configuration (GPU type, count, precision, TP degree).

In [ ]:
def recommend_config(model: Model, target_latency_ms: float, budget_per_hour: Optional[float] = None):    """Recommend optimal serving config given constraints."""    gpus = [A10G, A100_80, H100_80, RTX_6000]    precisions = [("FP16", 2), ("INT8", 1), ("INT4", 0.5)]    tp_options = [1, 2, 4, 8]    candidates = []    for gpu in gpus:        for prec_name, bpp in precisions:            for tp in tp_options:                weight_gb = model.params_b * 1e9 * bpp / 1e9                per_gpu_gb = weight_gb / tp                # Skip if doesn't fit                if per_gpu_gb > gpu.hbm_gb * 0.80:                    continue                # Estimate decode latency                bytes_to_read = model.params_b * 1e9 * bpp / tp                latency = (bytes_to_read / (gpu.bandwidth_gb_s * 1e9)) * 1000                # All-reduce overhead                if tp > 1:                    latency *= 1.05  # ~5% overhead approximation                if latency > target_latency_ms:                    continue                cost = tp * gpu.cost_per_hour                if budget_per_hour and cost > budget_per_hour:                    continue                candidates.append({                    "gpu": gpu.name, "count": tp, "precision": prec_name,                    "latency_ms": latency, "cost_hr": cost,                    "tps_batch1": 1000 / latency,                    "headroom_pct": (1 - latency / target_latency_ms) * 100                })    # Sort by cost, then latency    candidates.sort(key=lambda x: (x["cost_hr"], x["latency_ms"]))    return candidates# Example: Serve Llama-70B at < 30ms/token decode latencyprint("=" * 80)print("QUERY: Llama-2-70B, target < 30ms/token, no budget constraint")print("=" * 80)recs = recommend_config(LLAMA_70B, target_latency_ms=30.0)print(f"\n{'Rank':>4} | {'GPU':>14} | {'Count':>5} | {'Prec':>5} | {'Latency':>8} | {'TPS':>5} | {'$/hr':>6} | {'Headroom':>8}")print("-" * 80)for i, r in enumerate(recs[:8], 1):    print(f"{i:>4} | {r['gpu']:>14} | {r['count']:>5} | {r['precision']:>5} | "          f"{r['latency_ms']:>6.1f}ms | {r['tps_batch1']:>5.0f} | ${r['cost_hr']:>5.2f} | {r['headroom_pct']:>6.1f}%")print(f"\nBest pick: {recs[0]['count']}x {recs[0]['gpu']} @ {recs[0]['precision']} "      f"-- {recs[0]['latency_ms']:.1f}ms latency, ${recs[0]['cost_hr']:.2f}/hr")# Budget-constrained exampleprint("\n" + "=" * 80)print("QUERY: Llama-2-70B, target < 50ms/token, budget $6/hr")print("=" * 80)recs2 = recommend_config(LLAMA_70B, target_latency_ms=50.0, budget_per_hour=6.0)for i, r in enumerate(recs2[:5], 1):    print(f"{i}. {r['count']}x {r['gpu']} @ {r['precision']} -- "          f"{r['latency_ms']:.1f}ms, ${r['cost_hr']:.2f}/hr, {r['headroom_pct']:.0f}% headroom")